In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
import pandas as pd

demo = pd.read_csv('data/raw/df_final_demo.txt')
experiment = pd.read_csv('data/raw/df_final_experiment_clients.txt')

web_1 = pd.read_csv('data/raw/df_final_web_data_pt_1 .txt')
web_2 = pd.read_csv('data/raw/df_final_web_data_pt_2.txt')

In [ ]:
print(demo.shape)
print(experiment.shape)
print(web_1.shape)
print(web_2.shape)

In [ ]:
demo.head()

In [ ]:
web_1.head()

In [ ]:
experiment.head()

In [ ]:
web_2.head()

In [ ]:
demo.info()

In [ ]:
experiment.info()

In [ ]:
web_1.info()

In [ ]:
print("Demo duplicates:", demo.duplicated().sum())
print("Experiment duplicates:", experiment.duplicated().sum())
print("Web 1 duplicates:", web_1.duplicated().sum())
print("Web 2 duplicates:", web_2.duplicated().sum())

In [ ]:
#merge two datasets(pt1 and pt2)
web = pd.concat([web_1, web_2], ignore_index=True)
web.shape

In [ ]:
web.duplicated().sum()

In [ ]:
web = web.drop_duplicates()
web.shape

In [ ]:
demo.isnull().sum()

In [ ]:
experiment.isnull().sum()

In [ ]:
web.isnull().sum()

In [ ]:
demo['gendr'].value_counts(dropna=False)

In [ ]:
demo = demo[demo['gendr'] != 'X']
#Removed due to unclear category and negligible volume

In [ ]:
#clients without experiment assignment
experiment['Variation'].isnull().sum()

In [ ]:
#remove clients with no experiment assignement
experiment = experiment.dropna(subset=['Variation'])

In [ ]:
#find numerical variables
demo.describe()

In [ ]:
#find categorical vaiables

In [ ]:
#gender distribution 
demo['gendr'].value_counts(dropna=False)

In [ ]:
#experiment groups
experiment['Variation'].value_counts(dropna=False)

In [ ]:
#process steps
web['process_step'].value_counts()

In [ ]:
#visual EDA

In [ ]:
#age distribution 
plt.figure(figsize=(8,5))
plt.hist(demo['clnt_age'].dropna(), bins=30)
plt.title('Client Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Age Distribution Insights
#The online process is mainly used by middle-aged and older clients, especially those between 45 and 60 years old.
#Since older users may experience more usability friction, clearer navigation and interface design could improve completion rates and overall user 
#experience.

In [ ]:
#account balance distribution 
plt.figure(figsize=(8,5))
plt.hist(demo['bal'].dropna(), bins=30)
plt.title('Balance Distribution')
plt.xlabel('Balance')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(np.log1p(demo['bal'].dropna()), bins=30)
plt.title('Log-Transformed Balance Distribution')
plt.xlabel('Log(Balance)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Balance Distribution Insights
#The balance distribution is highly right-skewed. Most clients have moderate balances, while a small number of clients hold very large account balances.
#This suggests Vanguard serves both regular retail investors and some high-value clients

In [ ]:
#check unique clients
print("Unique clients in demo:", demo['client_id'].nunique())
print("Unique clients in experiment:", experiment['client_id'].nunique())
print("Unique clients in web:", web['client_id'].nunique())

In [ ]:
#convert datetime column
web['date_time'] = pd.to_datetime(web['date_time'])

In [ ]:
#check date ranges
print(web['date_time'].min())
print(web['date_time'].max())

In [ ]:
#session exploration 
web['visitor_id'].nunique()

In [ ]:
demo.columns

In [ ]:
#analyse tenure
demo['clnt_tenure_yr'].describe()

In [ ]:
#visualize tenure
plt.figure(figsize=(8,5))
plt.hist(demo['clnt_tenure_yr'].dropna(), bins=30)
plt.title('Client Tenure Distribution')
plt.xlabel('Years with Vanguard')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Client Tenure Insights
#Most users are long-term Vanguard clients, with many having between 5 and 20 years of tenure.
#This suggests the platform is mainly used by experienced and loyal investors, making improvements in the redesigned interface especially meaningful.

In [ ]:
#analyze digital activity
demo['logons_6_mnth'].describe()

In [ ]:
#visualize logoins
plt.figure(figsize=(8,5))
plt.hist(demo['logons_6_mnth'].dropna(), bins=30)
plt.title('Logons in Last 6 Months')
plt.xlabel('Number of Logons')
plt.ylabel('Frequency')
plt.show()

In [ ]:
#Digital Engagement Insights

#The logon distribution indicates that many clients interact with Vanguard’s online platform frequently, with most users logging in multiple times 
#during the previous six months.

#This suggests that the experiment primarily involves an already digitally engaged customer base rather than first-time online users. Therefore, 
#improvements in completion behavior are more likely attributable to the redesigned interface itself rather than increased familiarity with digital
#tools.

In [ ]:
#detect outliers
plt.figure(figsize=(8,5))
plt.boxplot(demo['bal'].dropna())
plt.title('Balance Outliers')
plt.show()

In [ ]:
#age outliers
plt.figure(figsize=(8,5))
plt.boxplot(demo['clnt_age'].dropna())
plt.title('Age Outliers')
plt.show()

In [ ]:
#balance outliers
Q1 = demo['bal'].quantile(0.25)
Q3 = demo['bal'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print(lower)
print(upper)

In [ ]:
outliers = demo[
    (demo['bal'] < lower) |
    (demo['bal'] > upper)
]
outliers.shape

In [ ]:
demo.describe()

In [ ]:
#Outlier Analysis

#Outliers were explored using descriptive statistics and boxplots. High balance values were retained because they likely represent real high-value 
#clients rather than data errors. Most variables showed realistic ranges, although a small number of unusually young clients may optionally be 
#excluded from the analysis.

In [ ]:
demo.isnull().all(axis=1).sum()

In [ ]:
experiment.isnull().all(axis=1).sum()

In [ ]:
web.isnull().all(axis=1).sum()

In [ ]:
experiment['Variation'].isnull().sum()

In [ ]:
df = ( web
    .merge(experiment, on='client_id', how='left')
    .merge(demo, on='client_id', how='left'))

In [ ]:
df_exp = df[
    df['Variation'].isin(['Control', 'Test'])]

## KPI Analysis

In [ ]:
#KPI 1 - Completion Rate ( clients reaching confirm ) 
completion = ( df_exp
    .groupby(['client_id', 'Variation'])['process_step']
    .apply(lambda x: 'confirm' in x.values)
    .reset_index(name='completed'))
completion.head()

In [ ]:
#Completion Rate by Group
completion_rate = (completion
    .groupby('Variation')
    .agg(
        total_clients=('client_id', 'nunique'),
        completed_clients=('completed', 'sum')
    ).reset_index())

completion_rate['completion_rate'] = (
    completion_rate['completed_clients']
    / completion_rate['total_clients'])
completion_rate

In [ ]:
#Visualize Completion Rate 
plt.figure(figsize=(6,4))
plt.bar(
    completion_rate['Variation'],
    completion_rate['completion_rate'])
plt.title('Completion Rate by Variation')
plt.ylabel('Completion Rate')
plt.show()

In [ ]:
#KPI 2 — Time Spent on Each Step ( seconds between consecutive events in same visit) 
#sort 
df_time = df_exp.sort_values(['client_id', 'visit_id', 'date_time']).copy()

In [ ]:
#calculate step duration 
df_time = df_exp.sort_values(['client_id', 'visit_id', 'date_time']).copy()

df_time['next_time'] = (
    df_time
    .groupby(['client_id', 'visit_id'])['date_time']
    .shift(-1)
)

df_time['step_duration_sec'] = (
    df_time['next_time'] - df_time['date_time']
).dt.total_seconds()

In [ ]:
#Remove Negative / Extreme Durations
df_time = df_time[
    (df_time['step_duration_sec'] >= 0) &
    (df_time['step_duration_sec'] <= 1800)
]

In [ ]:
#Average Time Per Step
step_time = (
    df_time
    .groupby(['Variation', 'process_step'])
    .agg(avg_time_sec=('step_duration_sec', 'mean'))
    .reset_index()
)

step_time

In [ ]:
#Visualize Step Duration
import seaborn as sns
plt.figure(figsize=(10,5))
sns.barplot( data=step_time, x='process_step', y='avg_time_sec', hue='Variation')
plt.title('Average Time Spent per Step')
plt.ylabel('Average Seconds')
plt.show()

In [ ]:
#KPI 3 — Error Rate (Backtracking, step visits where users moved to a previous step ) 
#define step order 
step_order = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}

In [ ]:
df_exp = df_exp.sort_values(['client_id', 'visit_id', 'date_time']).copy()

In [ ]:
#convert steps to numeric
df_exp['step_num'] = (df_exp['process_step'].map(step_order))

In [ ]:
df_exp['previous_step'] = (df_exp
    .groupby(['client_id', 'visit_id'])['step_num']
    .shift(1))
df_exp['backtrack'] = (
    (df_exp['step_num'] < df_exp['previous_step']) &
    (df_exp['step_num'].notna()) &
    (df_exp['previous_step'].notna())
)

In [ ]:
#error rate by group
error_rate = (df_exp
    .groupby('Variation')
    .agg(
        total_actions=('backtrack', 'count'),
        backtracks=('backtrack', 'sum') ) .reset_index())
error_rate['error_rate'] = (error_rate['backtracks']/ error_rate['total_actions'])
error_rate

In [ ]:
#Visualize error rate 
plt.figure(figsize=(6,4))
plt.bar(error_rate['Variation'],error_rate['error_rate'])
plt.title('Backtracking Error Rate')
plt.ylabel('Error Rate')
plt.show()

In [ ]:
# Percentage of visits with at least one backtracking error
# Identify if every visit had an error
visit_error_rate = (
    df_exp
    .groupby(['Variation', 'visit_id'])
    .agg(has_error=('backtrack', 'max'))
    .reset_index()
)


In [ ]:
# Group previous info (Control and Test)
visit_error_rate_summary = (
    visit_error_rate
    .groupby('Variation')
    .agg(
        total_visits=('visit_id', 'count'),
        visits_with_error=('has_error', 'sum')
    )
    .reset_index()
)

In [ ]:
#Get each percentage

visit_error_rate_summary['visit_error_rate'] = (
    visit_error_rate_summary['visits_with_error'] /
    visit_error_rate_summary['total_visits']
)

In [ ]:
# Visualize visit-level error rate

plt.figure(figsize=(6,4))
plt.bar(
    visit_error_rate_summary['Variation'],
    visit_error_rate_summary['visit_error_rate']
)

plt.title('Visit-Level Backtracking Error Rate')
plt.xlabel('Variation')
plt.ylabel('Visit Error Rate')
plt.ylim(0, visit_error_rate_summary['visit_error_rate'].max() * 1.2)

plt.show()

In [ ]:
#KPI 4 - Drop off Rate ( % of users who started but did not confirm or lost between each consecutive step)
# Drop-off Rate =(users who started but did not confirm)/ users who started

# Visits that started but did not reach confirmation
visit_steps = (
    df_exp
    .groupby(['Variation', 'visit_id'])['process_step']
    .agg(list)
    .reset_index()
)

visit_steps.head()



In [ ]:
# Check if visit reached confirmation

visit_steps['completed'] = (
    visit_steps['process_step']
    .apply(lambda x: 'confirm' in x)
)

visit_steps.head()

In [ ]:
# Calculate drop-off rate by group

dropoff_summary = (
    visit_steps
    .groupby('Variation')
    .agg(
        total_visits=('visit_id', 'count'),
        completed_visits=('completed', 'sum')
    )
    .reset_index()
)

dropoff_summary['dropoff_visits'] = (
    dropoff_summary['total_visits'] -
    dropoff_summary['completed_visits']
)

dropoff_summary['dropoff_rate'] = (
    dropoff_summary['dropoff_visits'] /
    dropoff_summary['total_visits']
)

dropoff_summary

In [ ]:
# Visualize overall drop-off rate

plt.figure(figsize=(6,4))

plt.bar(
    dropoff_summary['Variation'],
    dropoff_summary['dropoff_rate']
)

plt.title('Overall Drop-off Rate')
plt.xlabel('Variation')
plt.ylabel('Drop-off Rate')

plt.ylim(0, dropoff_summary['dropoff_rate'].max() * 1.2)

plt.show()

The redesigned interface reduced the overall drop-off rate. Control users had a drop-off rate of 50.15%, while Test users had a lower drop-off rate of 41.48%. This suggests that the new digital experience helped more users continue through the process and reach confirmation.

In [ ]:
# KPI 4 — Step-by-step Funnel Drop-off

funnel_steps = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

In [ ]:
#Count of visits by steps
funnel_counts = (
    df_exp[df_exp['process_step'].isin(funnel_steps)]
    .groupby(['Variation', 'process_step'])
    .agg(visits=('visit_id', 'nunique'))
    .reset_index()
)

funnel_counts

In [ ]:
#Count of visits by steps and by groups (Control vs Test)
funnel_counts['process_step'] = pd.Categorical(
    funnel_counts['process_step'],
    categories=funnel_steps,
    ordered=True
)

funnel_counts = funnel_counts.sort_values(['Variation', 'process_step'])

funnel_counts

In [ ]:
#Calculate visits from previous step
funnel_counts['previous_visits'] = (
    funnel_counts
    .groupby('Variation')['visits']
    .shift(1)
)

funnel_counts

In [ ]:
#Calculate drop-offs between steps
funnel_counts['step_dropoff'] = (
    funnel_counts['previous_visits'] - funnel_counts['visits']
)

funnel_counts['step_dropoff_rate'] = (
    funnel_counts['step_dropoff'] / funnel_counts['previous_visits']
)

funnel_counts

In [ ]:
# Visualize funnel visits by step

plt.figure(figsize=(8,5))

for variation in funnel_counts['Variation'].unique():
    data = funnel_counts[funnel_counts['Variation'] == variation]
    plt.plot(
        data['process_step'],
        data['visits'],
        marker='o',
        label=variation
    )

plt.title('Funnel Visits by Step')
plt.xlabel('Process Step')
plt.ylabel('Unique Visits')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Visualize step-by-step drop-off rate

dropoff_steps = funnel_counts.dropna(subset=['step_dropoff_rate'])

plt.figure(figsize=(8,5))

for variation in dropoff_steps['Variation'].unique():
    data = dropoff_steps[dropoff_steps['Variation'] == variation]
    plt.plot(
        data['process_step'],
        data['step_dropoff_rate'],
        marker='o',
        label=variation
    )

plt.title('Step-by-Step Drop-off Rate')
plt.xlabel('Process Step')
plt.ylabel('Drop-off Rate from Previous Step')
plt.legend()
plt.grid(True)

plt.show()

The redesigned interface significantly improved funnel retention. The Test group experienced lower drop-off rates across most funnel transitions, particularly in the final conversion stage from step_3 to confirmation, where abandonment dropped dramatically compared to the Control group. This suggests that the redesigned experience created a smoother and more effective completion flow.

In [ ]:
# KPI 5 — Repeat Step Rate
""" KPI 5 - Repeat Step Rate (users repeating the same step multiple times) could indicate confusion, 
unclear instructions and technical friction """

# Users repeating the same step multiple times

df_repeat = df_exp.sort_values(
    ['client_id', 'visit_id', 'date_time']
).copy()

In [ ]:
# Previous step within same visit

df_repeat['previous_step_same_visit'] = (
    df_repeat
    .groupby(['client_id', 'visit_id'])['process_step']
    .shift(1)
)

df_repeat.head()

In [ ]:
# Detect repeated steps

df_repeat['repeated_step'] = (
    df_repeat['process_step'] ==
    df_repeat['previous_step_same_visit']
)

df_repeat.head()

In [ ]:
# Calculate repeat step rate by group

repeat_step_rate = (
    df_repeat
    .groupby('Variation')
    .agg(
        total_actions=('repeated_step', 'count'),
        repeated_actions=('repeated_step', 'sum')
    )
    .reset_index()
)

repeat_step_rate['repeat_step_rate'] = (
    repeat_step_rate['repeated_actions'] /
    repeat_step_rate['total_actions']
)

repeat_step_rate

In [ ]:
# Visualize repeat step rate

plt.figure(figsize=(6,4))

plt.bar(
    repeat_step_rate['Variation'],
    repeat_step_rate['repeat_step_rate']
)

plt.title('Repeat Step Rate')
plt.xlabel('Variation')
plt.ylabel('Repeat Step Rate')

plt.ylim(0, repeat_step_rate['repeat_step_rate'].max() * 1.2)

plt.show()

The Test group showed a slightly higher repeat step rate compared to the Control group. This suggests that users interacting with the redesigned interface repeated the same steps more frequently, potentially indicating confusion, unclear guidance, or interaction friction within certain stages of the process.

In [ ]:
#Due to previous result, review where are people going back mostly from each group

# Repeat step rate by process step

repeat_by_step = (
    df_repeat
    .groupby(['Variation', 'process_step'])
    .agg(
        total_actions=('repeated_step', 'count'),
        repeated_actions=('repeated_step', 'sum')
    )
    .reset_index()
)

repeat_by_step['repeat_rate'] = (
    repeat_by_step['repeated_actions'] /
    repeat_by_step['total_actions']
)

repeat_by_step

In [ ]:
# Visualize repeat rate by process step

plt.figure(figsize=(9,5))

for variation in repeat_by_step['Variation'].unique():
    
    data = repeat_by_step[
        repeat_by_step['Variation'] == variation
    ]
    
    plt.plot(
        data['process_step'],
        data['repeat_rate'],
        marker='o',
        label=variation
    )

plt.title('Repeat Step Rate by Process Step')
plt.xlabel('Process Step')
plt.ylabel('Repeat Rate')

plt.legend()
plt.grid(True)

plt.show()

The redesigned experience improved conversion performance and reduced funnel abandonment. However, users in the Test group showed significantly higher repetition rates at the confirmation stage, suggesting uncertainty or insufficient feedback after completing the process. This indicates that while the redesign improved overall flow efficiency, the final confirmation interaction may require UX refinements to reduce repeated actions and user hesitation.

In [ ]:
#KPI 6  - Repeat visit Rate ( % of clients with more than 1 visit session ) 

# Percentage of clients with more than one visit session

client_visits = (
    df_exp
    .groupby(['Variation', 'client_id'])
    .agg(
        total_visits=('visit_id', 'nunique')
    )
    .reset_index()
)

client_visits.head()

In [ ]:
# Flag clients with more than one visit

client_visits['repeat_visitor'] = (
    client_visits['total_visits'] > 1
)

client_visits.head()

In [ ]:
# Calculate repeat visit rate by group

repeat_visit_rate = (
    client_visits
    .groupby('Variation')
    .agg(
        total_clients=('client_id', 'count'),
        repeat_visitors=('repeat_visitor', 'sum')
    )
    .reset_index()
)

repeat_visit_rate['repeat_visit_rate'] = (
    repeat_visit_rate['repeat_visitors'] /
    repeat_visit_rate['total_clients']
)

repeat_visit_rate

In [ ]:
# Visualize repeat visit rate

plt.figure(figsize=(6,4))

plt.bar(
    repeat_visit_rate['Variation'],
    repeat_visit_rate['repeat_visit_rate']
)

plt.title('Repeat Visit Rate')
plt.xlabel('Variation')
plt.ylabel('Repeat Visit Rate')

plt.ylim(0, repeat_visit_rate['repeat_visit_rate'].max() * 1.2)

plt.show()

Some users interacted with the redesigned experience across multiple sessions while still achieving higher completion outcomes overall.

## Conclusion

The redesigned digital experience improved overall process completion and reduced abandonment throughout the funnel. Users in the Test group progressed more successfully through each stage and reached confirmation more frequently than users in the Control group.

However, behavioral metrics such as higher backtracking rates, increased repeated step interactions, and slightly higher repeat visit rates suggest that some parts of the redesigned experience may still generate friction or uncertainty for users.

In particular, the confirmation stage showed a significantly higher repetition rate in the Test group, indicating that users may require clearer feedback or confirmation cues at the end of the process.

Overall, the redesign appears successful from a conversion perspective, but further UX refinements could improve navigation clarity and reduce unnecessary repeated interactions.

# Hypotheses Testing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import mannwhitneyu

In [ ]:
df_hyp = df_exp.copy() #The dataframe for the hypothesis to avoid changing the finished one

Significance level: α = 0.05

In [ ]:
df_hyp.head()

## Completion rate 

### Hypothesis

H0: The completion rate is the same in the Control and Test groups.

H1: The completion rate is different between the Control and Test groups.

In [ ]:
completion = (
    df_hyp
    .groupby(['Variation', 'client_id'])
    .agg(
        completed=('process_step', lambda x: 'confirm' in x.values)
    )
    .reset_index()
)

completion.head()

In [ ]:
#Group them by control or test

completion_summary = (
    completion
    .groupby('Variation')
    .agg(
        total_clients=('client_id', 'count'),
        completed_clients=('completed', 'sum')
    )
    .reset_index()
)

In [ ]:
#Calculate completion rate

completion_summary['completion_rate'] = (
    completion_summary['completed_clients'] /
    completion_summary['total_clients']
)

completion_summary

In [ ]:
#Proportions z-test since we are comparing proportions (completion rate)

count = completion_summary['completed_clients'].values
nobs = completion_summary['total_clients'].values

z_stat, p_value = proportions_ztest(count, nobs)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("Reject H0: The difference in completion rate is statistically significant.")
else:
    print("Fail to reject H0: The difference in completion rate is not statistically significant.")

### Visualization

In [ ]:
plt.figure(figsize=(6,4))

plt.bar(
    completion_summary['Variation'],
    completion_summary['completion_rate']
)

plt.title('Completion Rate by Group')
plt.xlabel('Variation')
plt.ylabel('Completion Rate')
plt.ylim(0, completion_summary['completion_rate'].max() * 1.2)

plt.show()

### Statistical Conclusion

The completion rate difference between the Control and Test groups is statistically significant (p < 0.05).

Therefore, we reject the null hypothesis and conclude that the redesign had a significant impact on the completion rate.

In [ ]:
# Double-checking how much improvement or difference we got

control_rate = completion_summary.loc[
    completion_summary['Variation'] == 'Control',
    'completion_rate'
].iloc[0]

test_rate = completion_summary.loc[
    completion_summary['Variation'] == 'Test',
    'completion_rate'
].iloc[0]

absolute_lift = test_rate - control_rate

relative_lift = absolute_lift / control_rate

print(f"Control Completion Rate: {control_rate:.2%}")
print(f"Test Completion Rate: {test_rate:.2%}")
print(f"Absolute Lift: {absolute_lift:.2%}")
print(f"Relative Lift: {relative_lift:.2%}")

## Completion rate with the cost-effect

### Hypothesis

H0: 

H1: 

H0 :The completion rate improvement is less than or equal to 5%.
H1 :The completion rate improvement is greater than 5%.

In [ ]:
alpha = 0.05

In [ ]:
print(f"Control Completion Rate: {control_rate:.2%}")
print(f"Test Completion Rate: {test_rate:.2%}")
print(f"Absolute Lift: {absolute_lift:.2%}")